In [1]:
import os
os.environ['KAGGLE_API_TOKEN'] = "KGAT_1e904cbdf781453639a1c8cc6334dd3c"
!pip install --upgrade kaggle -q
print("Downloading BTD Dataset (This might take a few minutes depending on Colab's speed)...")
!kaggle datasets download -d freddiegraboski/btd-mri-and-ct-deepfake-test-sets
print("Unzipping data...")
!unzip -q btd-mri-and-ct-deepfake-test-sets.zip -d btd_dataset
print("Dataset successfully loaded into Colab!")

Dataset URL: https://www.kaggle.com/datasets/freddiegraboski/btd-mri-and-ct-deepfake-test-sets
License(s): GNU Affero General Public License 3.0
100% 1.25G/1.25G [00:33<00:00, 40.6MB/s]

Unzipping data...
Dataset successfully loaded into Colab!


In [2]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from google.colab import files
import warnings

warnings.filterwarnings("ignore")

class MedicalScanDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_paths = []
        for ext in ('*.png', '*.jpg', '*.jpeg'):
            self.image_paths.extend(glob.glob(os.path.join(root_dir, '**', ext), recursive=True))
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        filename = os.path.basename(img_path).lower()
        is_fake = any(word in filename for word in ['fake', 'injection', 'removal'])
        label = 1 if is_fake else 0 # 1 = Fake, 0 = Real

        if self.transform:
            image = self.transform(image)

        return image, label

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} (If this says 'cpu', stop and change runtime to T4 GPU!)")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Scanning folders for medical images...")
dataset = MedicalScanDataset(root_dir='/content/btd_dataset', transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)
print(f"Found {len(dataset)} images for training.")

print("Loading pre-trained ResNet-18 base...")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop (3 Epochs)
epochs = 3
print("\n--- Starting Training ---")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for i, (inputs, labels) in enumerate(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        if i % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{i}/{len(dataloader)}], Loss: {loss.item():.4f}")

    epoch_acc = 100 * correct / total
    print(f"-> Epoch {epoch+1} Completed. Accuracy: {epoch_acc:.2f}%\n")

model_save_path = "medical_deepfake_resnet18.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Model successfully saved to {model_save_path}!")

print("Downloading the customized weights to your laptop...")
files.download(model_save_path)

Using device: cuda (If this says 'cpu', stop and change runtime to T4 GPU!)
Scanning folders for medical images...
Found 525 images for training.
Loading pre-trained ResNet-18 base...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 170MB/s]



--- Starting Training ---
Epoch [1/3], Step [0/17], Loss: 0.6019
Epoch [1/3], Step [10/17], Loss: 0.6351
-> Epoch 1 Completed. Accuracy: 60.95%

Epoch [2/3], Step [0/17], Loss: 0.6588
Epoch [2/3], Step [10/17], Loss: 0.6341
-> Epoch 2 Completed. Accuracy: 68.19%

Epoch [3/3], Step [0/17], Loss: 0.4851
Epoch [3/3], Step [10/17], Loss: 0.4904
-> Epoch 3 Completed. Accuracy: 66.10%

Model successfully saved to medical_deepfake_resnet18.pth!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from google.colab import files
import warnings

warnings.filterwarnings("ignore")

class MedicalScanDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_paths = []
        for ext in ('*.png', '*.jpg', '*.jpeg'):
            self.image_paths.extend(glob.glob(os.path.join(root_dir, '**', ext), recursive=True))
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')

        filename = os.path.basename(img_path).lower()
        is_fake = any(word in filename for word in ['fake', 'injection', 'removal'])
        label = 1 if is_fake else 0

        if self.transform:
            image = self.transform(image)
        return image, label

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data Augmentation added
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = MedicalScanDataset(root_dir='/content/btd_dataset', transform=train_transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 20 Epochs
epochs = 20
print("\n--- Starting High-Accuracy Training ---")
for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(dataloader):.4f} | Accuracy: {epoch_acc:.2f}%")

model_save_path = "medical_deepfake_resnet18_v2.pth"
torch.save(model.state_dict(), model_save_path)
print("Downloading V2 weights to your laptop...")
files.download(model_save_path)


--- Starting High-Accuracy Training ---
Epoch 1/20 | Loss: 0.7875 | Accuracy: 62.10%
Epoch 2/20 | Loss: 0.5244 | Accuracy: 69.71%
Epoch 3/20 | Loss: 0.5576 | Accuracy: 70.86%
Epoch 4/20 | Loss: 0.4769 | Accuracy: 75.05%
Epoch 5/20 | Loss: 0.3858 | Accuracy: 81.33%
Epoch 6/20 | Loss: 0.3287 | Accuracy: 85.90%
Epoch 7/20 | Loss: 0.3136 | Accuracy: 86.10%
Epoch 8/20 | Loss: 0.4054 | Accuracy: 82.10%
Epoch 9/20 | Loss: 0.2992 | Accuracy: 85.90%
Epoch 10/20 | Loss: 0.2542 | Accuracy: 87.62%
Epoch 11/20 | Loss: 0.2208 | Accuracy: 89.52%
Epoch 12/20 | Loss: 0.2578 | Accuracy: 87.24%
Epoch 13/20 | Loss: 0.2321 | Accuracy: 90.48%
Epoch 14/20 | Loss: 0.1714 | Accuracy: 93.33%
Epoch 15/20 | Loss: 0.2028 | Accuracy: 91.81%
Epoch 16/20 | Loss: 0.1360 | Accuracy: 94.86%
Epoch 17/20 | Loss: 0.1122 | Accuracy: 96.57%
Epoch 18/20 | Loss: 0.2936 | Accuracy: 87.24%
Epoch 19/20 | Loss: 0.2314 | Accuracy: 90.29%
Epoch 20/20 | Loss: 0.1631 | Accuracy: 92.95%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import os
import glob
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from google.colab import files
import warnings

warnings.filterwarnings("ignore")

class MedicalScanDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_paths = []
        for ext in ('*.png', '*.jpg', '*.jpeg'):
            self.image_paths.extend(glob.glob(os.path.join(root_dir, '**', ext), recursive=True))
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')

        filename = os.path.basename(img_path).lower()
        is_fake = any(word in filename for word in ['fake', 'injection', 'removal'])
        label = 1 if is_fake else 0

        if self.transform:
            image = self.transform(image)
        return image, label

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Advanced Data Augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = MedicalScanDataset(root_dir='/content/btd_dataset', transform=train_transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)

# ResNet-50 (Deeper Architecture)
print("Loading heavy-duty ResNet-50 base...")
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning Rate Scheduler (Slows down by 10x every 7 epochs)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

epochs = 25
best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0

print("\n--- Starting Ultimate Accuracy Training ---")
for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()


    exp_lr_scheduler.step()

    epoch_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(dataloader):.4f} | Accuracy: {epoch_acc:.2f}%")


    if epoch_acc > best_acc:
        best_acc = epoch_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        print(f"🌟 New High Score! Checkpointing model at {best_acc:.2f}%")

print(f"\nTraining complete! Peak accuracy reached: {best_acc:.2f}%")

model.load_state_dict(best_model_wts)
model_save_path = "medical_deepfake_resnet50_ULTIMATE.pth"
torch.save(model.state_dict(), model_save_path)

print("Downloading the ultimate weights to your laptop...")
files.download(model_save_path)


Loading heavy-duty ResNet-50 base...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 183MB/s]



--- Starting Ultimate Accuracy Training ---
Epoch 1/25 | Loss: 0.6442 | Accuracy: 63.24%
🌟 New High Score! Checkpointing model at 63.24%
Epoch 2/25 | Loss: 0.5821 | Accuracy: 66.67%
🌟 New High Score! Checkpointing model at 66.67%
Epoch 3/25 | Loss: 0.5698 | Accuracy: 67.05%
🌟 New High Score! Checkpointing model at 67.05%
Epoch 4/25 | Loss: 0.5918 | Accuracy: 67.43%
🌟 New High Score! Checkpointing model at 67.43%
Epoch 5/25 | Loss: 0.5451 | Accuracy: 70.48%
🌟 New High Score! Checkpointing model at 70.48%
Epoch 6/25 | Loss: 0.4309 | Accuracy: 77.90%
🌟 New High Score! Checkpointing model at 77.90%
Epoch 7/25 | Loss: 0.4239 | Accuracy: 79.05%
🌟 New High Score! Checkpointing model at 79.05%
Epoch 8/25 | Loss: 0.4449 | Accuracy: 78.48%
Epoch 9/25 | Loss: 0.2937 | Accuracy: 89.33%
🌟 New High Score! Checkpointing model at 89.33%
Epoch 10/25 | Loss: 0.3039 | Accuracy: 89.33%
Epoch 11/25 | Loss: 0.2429 | Accuracy: 89.90%
🌟 New High Score! Checkpointing model at 89.90%
Epoch 12/25 | Loss: 0.2324

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>